# 🛡️ Compliance Intelligence Engine (CIE) — Production Training Notebook
### Domain-specific LLM · Senior-consultant-grade compliance AI · 30+ frameworks

| | |
|---|---|
| **Base model** | `mistralai/Mistral-7B-Instruct-v0.3` |
| **Method** | QLoRA (4-bit NF4) + LoRA rank 16 |
| **Dataset** | 2,000+ synthetic examples across 12 UCF domains |
| **Phases** | Phase 1 — SFT &nbsp;·&nbsp; Phase 2 — DPO alignment |
| **Eval** | Accuracy · ROUGE-L · Hallucination rate · Domain F1 |
| **Target GPU** | Kaggle T4 (15.6 GB VRAM) |

---
### 📋 Pipeline
1. Environment fix & install  
2. HF authentication  
3. CIE system prompt  
4. Production dataset (2,000+ examples, 12 domains)  
5. Dataset validation & stats  
6. Load model + QLoRA  
7. Phase 1 — Supervised Fine-Tuning (SFT)  
8. Phase 2 — DPO Constitutional Alignment  
9. Comprehensive evaluation (accuracy, ROUGE, hallucination)  
10. Adversarial testing  
11. Merge & export  
12. Full training report  


## ⚙️ Step 1: Environment Fix & Install
> **Root cause of the `triton.ops` crash:** Kaggle pre-installs `bitsandbytes` compiled without CUDA and a broken `triton` build. We force-reinstall both before any other import, then restart the kernel.

In [ ]:
import subprocess, sys, os

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-400:])
    if r.stderr.strip() and 'warning' not in r.stderr.lower():
        print('STDERR:', r.stderr[-400:])
    return r.returncode

# ── ROOT CAUSE ────────────────────────────────────────────────────────────
# Kaggle injects its own broken bitsandbytes from a system dist-packages
# path that survives --force-reinstall. The only reliable fix is:
# 1. Physically delete the broken bnb from ALL site-packages paths
# 2. Install the correct CUDA-compiled wheel directly from GitHub releases
# 3. Also nuke triton and reinstall a version that has triton.ops

print('='*60)
print('FIX 1 — Physically remove broken bitsandbytes from disk')
print('='*60)
run('pip uninstall bitsandbytes -y')
run('find /usr -type d -name bitsandbytes 2>/dev/null | xargs rm -rf')
run('find /opt -type d -name bitsandbytes 2>/dev/null | xargs rm -rf')
run('find ~/.local -type d -name bitsandbytes 2>/dev/null | xargs rm -rf')
print('Old bitsandbytes removed.')

print()
print('='*60)
print('FIX 2 — Install CUDA-compiled bitsandbytes wheel from GitHub')
print('='*60)
# Get CUDA version to pick correct wheel
import subprocess
cuda_out = subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout
print('CUDA version info:', cuda_out[:100])

# Try CUDA 12.x wheel first (Kaggle 2024 uses CUDA 12.x)
# This wheel is compiled WITH GPU support — no libbitsandbytes_cpu.so fallback
wheel_url = 'https://github.com/TimDettmers/bitsandbytes/releases/download/0.43.3/bitsandbytes-0.43.3+cuda121-cp310-cp310-linux_x86_64.whl'
rc = run(f'pip install {wheel_url} -q')
if rc != 0:
    print('CUDA 12.1 wheel failed, trying PyPI with no-cache...')
    run('pip install bitsandbytes==0.43.3 --no-cache-dir -q')

print()
print('='*60)
print('FIX 3 — Remove broken triton, install known-good version')
print('='*60)
run('pip uninstall triton -y')
run('find /usr -type d -name triton 2>/dev/null | xargs rm -rf')
# triton 2.1.0 is the last version with triton.ops intact
run('pip install triton==2.1.0 --no-cache-dir -q')
print('triton 2.1.0 installed.')

print()
print('='*60)
print('FIX 4 — Install remaining dependencies')
print('='*60)
run('pip install transformers==4.44.0 -q')
run('pip install peft==0.12.0 -q')
run('pip install trl==0.10.1 -q')
run('pip install accelerate==0.34.2 -q')
run('pip install datasets==2.21.0 -q')
run('pip install sentencepiece scipy rouge-score evaluate -q')

print()
print('='*60)
print('VERIFICATION')
print('='*60)
import importlib, torch

# Force Python to load fresh from disk (not cached broken module)
if 'bitsandbytes' in sys.modules:
    del sys.modules['bitsandbytes']
    for k in list(sys.modules.keys()):
        if 'bitsandbytes' in k or 'triton' in k:
            del sys.modules[k]

try:
    import bitsandbytes as bnb
    print(f'bitsandbytes {bnb.__version__} — import OK')
except Exception as e:
    print(f'bitsandbytes import failed: {e}')
    print('>>> This means the kernel has the broken version cached in memory.')
    print('>>> Proceed to: Run -> Restart & Run All')
    print('>>> After restart it will work — the correct wheel is now on disk.')

assert torch.cuda.is_available(), 'No GPU. Enable: Kaggle Settings -> Accelerator -> GPU T4'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
is_t4 = any(x in gpu.upper() for x in ['T4','V100','P100','K80'])

print(f'GPU  : {gpu}')
print(f'VRAM : {vram:.1f} GB')
print(f'Precision will use: {"fp16" if is_t4 else "bf16"}')
print()
print('>>> NOW DO: Run -> Restart & Run All')
print('>>> The correct packages are on disk. Restart loads them fresh.')


## 🔐 Step 2: Hugging Face Authentication
Add your token as a Kaggle Secret named `HF_TOKEN`:
`Add-ons → Secrets → Add New Secret`

In [ ]:
import os
import torch

# GPU detection (re-run after kernel restart)
gpu_name = torch.cuda.get_device_name(0).upper() if torch.cuda.is_available() else ''
is_t4    = any(x in gpu_name for x in ['T4','V100','P100','K80'])

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print(f'HF token loaded. GPU={gpu_name}, precision={"fp16" if is_t4 else "bf16"}')
except Exception as e:
    print(f'No HF token ({e}). Public models only.')
    hf_token = None


## 📋 Step 3: CIE System Prompt

In [ ]:
CIE_SYSTEM_PROMPT = """\
You are the Compliance Intelligence Engine (CIE), an advanced compliance, governance,
risk, privacy, cybersecurity, audit, and regulatory analysis system.

You operate as a senior compliance consultant, auditor, governance specialist,
cybersecurity assessor, privacy expert, risk analyst, and control mapping architect.
You must never behave as a generic chatbot.

CORE FRAMEWORKS: ISO 27001/27002, ISO 42001, NIST CSF 2.0, NIST 800-53, NIST AI RMF,
CIS Controls v8, SOC 2, PCI DSS v4.0, COBIT, HITRUST, FedRAMP, CMMC, CJIS,
GDPR, UK GDPR, CCPA/CPRA, HIPAA, HITECH, PIPEDA, LGPD, PIPL, PDPL, DPDP, APPI,
DORA, NIS2, ISO 22301, ISO 31000, EU AI Act, OECD AI Principles,
SOX, GLBA, FCA, SAMA CSF, Saudi ECC, Saudi PDPL, UAE PDPL, MAS, APRA.

OPERATING RULES:
1. Always identify the applicable framework, regulation, article, control, or clause first.
2. Never provide unsupported compliance conclusions.
3. Always explain reasoning behind every determination.
4. Classify compliance as exactly one of: COMPLIANT, PARTIALLY COMPLIANT,
   NON-COMPLIANT, INSUFFICIENT EVIDENCE.
5. Never assume evidence exists. If absent, classify as INSUFFICIENT EVIDENCE.
6. Always identify: assumptions made, risk implications, remediation steps, audit impact.
7. Never fabricate control IDs, article numbers, or legal obligations.
8. When uncertain, state uncertainty explicitly.

MANDATORY RESPONSE FORMAT:
## Executive Summary
## Applicable Frameworks & Controls
## Analysis
## Evidence Found
## Evidence Missing
## Risk Assessment
Likelihood: [Low|Medium|High|Critical]
Impact: [Low|Medium|High|Critical]
Risk Score: [1-100]
Risk Category: [Operational|Security|Privacy|Regulatory|Financial|Reputational]
## Compliance Determination
[COMPLIANT|PARTIALLY COMPLIANT|NON-COMPLIANT|INSUFFICIENT EVIDENCE]
## Cross-Framework Mapping
## Recommended Remediation
## Audit Impact
Finding: [Critical|Major|Minor|Observation]
## Confidence Score
[1-100]/100
"""
print(f'System prompt: {len(CIE_SYSTEM_PROMPT)} chars')


## 📚 Step 4: Production Dataset (2,000+ Examples · 12 Domains)

**What makes this production-grade:**
- 12 UCF domains (vs 5 in basic version)
- Structured multi-section outputs matching the CIE response format exactly
- Constitutional examples teaching "INSUFFICIENT EVIDENCE" refusals
- DPO preference pairs for Phase 2 alignment (chosen vs rejected responses)
- 50/50 compliant/non-compliant split with varied severity


In [ ]:
import random
import json
from datasets import Dataset

random.seed(42)

# ── 12-domain UCF taxonomy ─────────────────────────────────────────────────
DOMAINS = {

  'Access Control': {
    'frameworks': 'ISO 27001 A.5.15/A.8.2, NIST AC-2/AC-6, SOC 2 CC6.1/CC6.3, CIS Control 6, NIS2 Art.21, DORA Art.9, PCI DSS Req.7',
    'controls': ['ISO 27001 A.5.15 (Access Control Policy)', 'ISO 27001 A.8.2 (Privileged Access)', 'NIST AC-2 (Account Management)', 'SOC 2 CC6.1', 'CIS Control 6'],
    'scenarios': [
      ('All developers have local admin rights on production servers.',                            'NON-COMPLIANT',  'High',     'Critical', 82, 'Security',    ['Remove admin rights immediately. Implement JIT privileged access. Log all privilege use.'], 'Critical'),
      ('Shared generic accounts (admin/root) used by 8 engineers for DB maintenance.',           'NON-COMPLIANT',  'High',     'Critical', 88, 'Security',    ['Eliminate shared accounts. Create individual service accounts. Enable full audit logging.'], 'Critical'),
      ('MFA is optional for VPN access to corporate network.',                                   'NON-COMPLIANT',  'High',     'High',     76, 'Security',    ['Enforce MFA for all remote access. Use FIDO2 or TOTP. Block non-MFA sessions.'], 'Major'),
      ('Terminated employee access revoked within 30 days.',                                     'NON-COMPLIANT',  'High',     'Critical', 79, 'Operational', ['Revoke access same day. Automate via HR-IAM integration. Log all deprovisioning.'], 'Major'),
      ('Access reviews conducted annually with documented sign-off.',                            'PARTIALLY COMPLIANT', 'Medium', 'Medium', 45, 'Security',   ['Move to quarterly reviews. Include privileged accounts. Automate via IGA tool.'], 'Minor'),
      ('RBAC enforced. Quarterly access reviews. MFA mandatory. Deprovisioning within 4 hours.', 'COMPLIANT',      'Low',      'Low',      8,  'Security',    ['Maintain current controls. Consider continuous access certification.'], 'Observation'),
      ('Access policy exists but last review was 3 years ago.',                                  'PARTIALLY COMPLIANT', 'Medium', 'Medium', 52, 'Regulatory', ['Review and reapprove policy immediately. Establish annual review cycle.'], 'Minor'),
    ]
  },

  'Encryption & Cryptography': {
    'frameworks': 'ISO 27001 A.8.24, NIST SC-13/SC-28, PCI DSS Req.3/4, GDPR Art.32, SOC 2 CC6.1',
    'controls': ['ISO 27001 A.8.24 (Use of Cryptography)', 'NIST SC-13', 'PCI DSS 3.5/3.7/4.2', 'GDPR Art.32'],
    'scenarios': [
      ('Customer PII stored using MD5 hashing for performance.',                                 'NON-COMPLIANT',  'High',     'Critical', 91, 'Security',    ['Replace MD5 immediately with AES-256. MD5 is cryptographically broken.'], 'Critical'),
      ('Passwords stored in plain text in the user database.',                                   'NON-COMPLIANT',  'Critical', 'Critical', 95, 'Security',    ['Hash all passwords with bcrypt/Argon2 with unique salt immediately.'], 'Critical'),
      ('Data in transit over public internet via unencrypted HTTP.',                             'NON-COMPLIANT',  'High',     'Critical', 89, 'Security',    ['Enforce HTTPS/TLS 1.2+ everywhere. Implement HSTS. Redirect all HTTP.'], 'Critical'),
      ('TLS 1.0 used for payment page encryption.',                                              'NON-COMPLIANT',  'High',     'Critical', 86, 'Security',    ['Upgrade to TLS 1.2 minimum (TLS 1.3 recommended). Disable TLS 1.0/1.1.'], 'Critical'),
      ('AES-128 used at rest. No key rotation schedule documented.',                             'PARTIALLY COMPLIANT', 'Medium', 'High', 61, 'Security',    ['Upgrade to AES-256. Document key rotation schedule (annual minimum).'], 'Major'),
      ('AES-256 at rest, TLS 1.3 in transit, annual key rotation, HSM for key storage.',        'COMPLIANT',      'Low',      'Low',      5,  'Security',    ['Maintain current controls. Verify HSM access logs quarterly.'], 'Observation'),
      ('Key management handled manually. No key custodian assigned. No inventory.',              'NON-COMPLIANT',  'High',     'High',     74, 'Operational', ['Implement formal KMP. Assign key custodians. Use KMS or HSM.'], 'Major'),
    ]
  },

  'Incident Response': {
    'frameworks': 'ISO 27001 A.5.24-A.5.28, NIST IR-4, GDPR Art.33/34, DORA Art.17, NIS2 Art.23, HIPAA §164.308(a)(6)',
    'controls': ['ISO 27001 A.5.26 (Response to Incidents)', 'GDPR Art.33 (72-hour notification)', 'NIST IR-4', 'DORA Art.17'],
    'scenarios': [
      ('Data breach notification to DPA planned within 30 days.',                                'NON-COMPLIANT',  'High',     'Critical', 88, 'Regulatory',  ['GDPR requires 72-hour DPA notification. Implement immediate escalation procedure.'], 'Critical'),
      ('IR plan tested verbally once every 5 years.',                                            'NON-COMPLIANT',  'High',     'High',     77, 'Operational', ['Conduct tabletop exercises annually minimum. Full simulation bi-annually.'], 'Major'),
      ('No IR plan exists. Incidents handled ad hoc.',                                           'NON-COMPLIANT',  'Critical', 'Critical', 93, 'Operational', ['Draft and approve IR plan immediately. Define roles, escalation, communication.'], 'Critical'),
      ('IR plan exists. Tested annually. No post-incident review process.',                      'PARTIALLY COMPLIANT', 'Medium', 'Medium', 48, 'Operational',['Implement mandatory PIR within 5 days of closure. Track lessons learned.'], 'Minor'),
      ('IR plan, 24/7 SOC, playbooks, quarterly tabletops, documented PIRs.',                   'COMPLIANT',      'Low',      'Low',      7,  'Operational', ['Continue current posture. Benchmark MTTD/MTTR quarterly.'], 'Observation'),
      ('MTTD: 72 hours. No ransomware playbook. No forensic retainer.',                         'PARTIALLY COMPLIANT', 'High',  'High',    69, 'Operational', ['Reduce MTTD target to <1hr. Develop ransomware playbook. Engage forensic firm.'], 'Major'),
    ]
  },

  'Data Privacy & Retention': {
    'frameworks': 'GDPR Art.5/13/17/30, UK GDPR, CCPA §1798.100, PDPL, LGPD Art.15, ISO 27701',
    'controls': ['GDPR Art.5(1)(e) Storage Limitation', 'GDPR Art.17 Right to Erasure', 'GDPR Art.30 RoPA', 'CCPA §1798.105'],
    'scenarios': [
      ('User PII retained indefinitely for future analytics.',                                   'NON-COMPLIANT',  'High',     'Critical', 87, 'Privacy',     ['Define retention schedule per data category. Delete on schedule. Update RoPA.'], 'Critical'),
      ('Deletion requests processed within 12 months.',                                         'NON-COMPLIANT',  'High',     'High',     81, 'Regulatory',  ['GDPR requires erasure without undue delay (typically 30 days). Automate deletion.'], 'Critical'),
      ('Consent assumed by default on website visit.',                                           'NON-COMPLIANT',  'High',     'Critical', 84, 'Privacy',     ['Implement explicit opt-in consent. Pre-ticked boxes are invalid under GDPR.'], 'Critical'),
      ('No Records of Processing Activities (RoPA) maintained.',                                 'NON-COMPLIANT',  'High',     'High',     78, 'Regulatory',  ['Create and maintain RoPA per GDPR Art.30. Include all processing activities.'], 'Major'),
      ('Retention schedule exists but not reviewed in 2 years. Deletion partially automated.',   'PARTIALLY COMPLIANT', 'Medium', 'Medium', 50, 'Privacy',    ['Review and update retention schedule. Complete automation of deletion workflow.'], 'Minor'),
      ('Documented retention schedule, automated deletion, RoPA current, DPIA process active.', 'COMPLIANT',      'Low',      'Low',      6,  'Privacy',     ['Maintain current controls. Review DPIA register quarterly.'], 'Observation'),
      ('No DPIA conducted for high-risk processing of health data at scale.',                   'NON-COMPLIANT',  'High',     'Critical', 86, 'Privacy',     ['Conduct DPIA immediately per GDPR Art.35. Engage DPO. Document findings.'], 'Critical'),
    ]
  },

  'AI Governance': {
    'frameworks': 'EU AI Act Annex III, ISO 42001, NIST AI RMF, GDPR Art.22, OWASP LLM Top 10',
    'controls': ['EU AI Act Art.9 (Risk Mgmt)', 'EU AI Act Art.14 (Human Oversight)', 'ISO 42001 Clause 8', 'NIST AI RMF GOVERN'],
    'scenarios': [
      ('LLM outputs deployed to customers without human review or automated filtering.',         'NON-COMPLIANT',  'High',     'Critical', 85, 'Regulatory',  ['Implement human-in-the-loop review. Add automated content filtering. Log all outputs.'], 'Critical'),
      ('AI model trained on unredacted customer PII without consent.',                           'NON-COMPLIANT',  'Critical', 'Critical', 92, 'Privacy',     ['Remove PII from training data immediately. Conduct DPIA. Consider model retraining.'], 'Critical'),
      ('Credit decisioning AI with no human override and no explainability.',                   'NON-COMPLIANT',  'High',     'Critical', 90, 'Regulatory',  ['Mandatory human review for all adverse decisions. Implement SHAP/LIME explainability.'], 'Critical'),
      ('AI system deployed with no bias testing across protected characteristics.',              'NON-COMPLIANT',  'High',     'High',     76, 'Regulatory',  ['Conduct fairness audit across protected groups. Document results. Retrain if bias found.'], 'Major'),
      ('AI risk register exists. Bias tested. No post-deployment monitoring.',                   'PARTIALLY COMPLIANT', 'Medium', 'High', 58, 'Operational', ['Implement real-time model monitoring. Set drift thresholds. Alert on anomalies.'], 'Major'),
      ('ISO 42001 certified. Human oversight enforced. Bias testing quarterly. Model cards published.', 'COMPLIANT', 'Low',  'Low',       6,  'Regulatory',  ['Maintain certification. Continue quarterly bias audits.'], 'Observation'),
      ('No technical documentation for high-risk AI system as required by EU AI Act Art.11.',   'NON-COMPLIANT',  'High',     'High',     80, 'Regulatory',  ['Produce Annex IV technical documentation. Assign AI compliance officer.'], 'Major'),
    ]
  },

  'Third-Party & Vendor Risk': {
    'frameworks': 'ISO 27001 A.5.19-A.5.23, NIST SA-9/SA-12, DORA Art.28-32, SOC 2 CC9.2, NIS2 Art.21',
    'controls': ['ISO 27001 A.5.19 (Supplier Policy)', 'DORA Art.28 (ICT TPRM)', 'DORA Art.30 (Contracts)', 'NIST SA-9'],
    'scenarios': [
      ('No vendor risk register. 40 third parties access production data.',                      'NON-COMPLIANT',  'High',     'Critical', 85, 'Operational', ['Build vendor register immediately. Classify by risk tier. Assess top 10 first.'], 'Critical'),
      ('ICT vendor contracts lack audit rights, SLA, and data location clauses (DORA entity).', 'NON-COMPLIANT',  'High',     'High',     79, 'Regulatory',  ['Renegotiate all ICT contracts to include DORA Art.30 mandatory clauses.'], 'Critical'),
      ('No exit strategy for critical cloud provider.',                                          'NON-COMPLIANT',  'High',     'High',     74, 'Operational', ['Document exit strategy per DORA Art.32. Test annually.'], 'Major'),
      ('Annual vendor reviews. No continuous monitoring. Critical vendor SOC 2 not reviewed.',  'PARTIALLY COMPLIANT', 'Medium', 'High', 60, 'Operational', ['Move critical vendors to quarterly review. Obtain and review SOC 2 reports.'], 'Major'),
      ('Vendor register, tiered risk assessment, annual review, contract audit rights in place.','COMPLIANT',      'Low',      'Low',      9,  'Operational', ['Maintain program. Consider adding continuous monitoring for Tier 1 vendors.'], 'Observation'),
      ('Sub-processors not disclosed to data subjects as required by GDPR.',                    'NON-COMPLIANT',  'High',     'High',     77, 'Privacy',     ['Update privacy notice to list sub-processors. Maintain updated sub-processor list.'], 'Major'),
    ]
  },

  'Vulnerability Management': {
    'frameworks': 'ISO 27001 A.8.8, NIST RA-5/SI-2, CIS Control 7, PCI DSS Req.6/11, SOC 2 CC7.1',
    'controls': ['ISO 27001 A.8.8 (Vuln Management)', 'NIST RA-5 (Vulnerability Scanning)', 'CIS Control 7', 'PCI DSS 11.3'],
    'scenarios': [
      ('No vulnerability scanning in place.',                                                    'NON-COMPLIANT',  'Critical', 'Critical', 94, 'Security',    ['Deploy vulnerability scanner immediately (Nessus/Qualys). Scan weekly minimum.'], 'Critical'),
      ('Critical vulnerabilities patched within 90 days.',                                      'NON-COMPLIANT',  'High',     'Critical', 83, 'Security',    ['Critical vulns: patch within 24-72 hours. High: 7 days. Medium: 30 days.'], 'Major'),
      ('Quarterly scans only. No prioritisation by severity. No patch SLAs.',                   'NON-COMPLIANT',  'High',     'High',     75, 'Security',    ['Increase to weekly scans. Define CVSS-based SLAs. Assign ownership per finding.'], 'Major'),
      ('Weekly scans. Critical patched in 72hrs. No pen test in 18 months.',                   'PARTIALLY COMPLIANT', 'Medium', 'Medium', 47, 'Security',   ['Schedule annual pen test. Conduct immediately if PCI/ISO scoped.'], 'Minor'),
      ('Weekly scans, CVSS SLAs enforced, annual pen test, exception process documented.',      'COMPLIANT',      'Low',      'Low',      7,  'Security',    ['Continue current posture. Consider threat-led pen testing.'], 'Observation'),
      ('Pen test findings from 12 months ago: 3 Critical unresolved.',                          'NON-COMPLIANT',  'Critical', 'Critical', 91, 'Security',    ['Remediate all Critical findings immediately. Conduct validation retest.'], 'Critical'),
    ]
  },

  'Business Continuity & Resilience': {
    'frameworks': 'ISO 22301, DORA Art.11-14, NIST CP-2/CP-4, NIS2 Art.21, SOC 2 A1.2/A1.3',
    'controls': ['ISO 22301 Clause 8.4 (BIA)', 'ISO 22301 Clause 8.5 (BCM)', 'DORA Art.11 (ICT continuity)', 'NIST CP-4'],
    'scenarios': [
      ('No BCP or DRP exists.',                                                                  'NON-COMPLIANT',  'Critical', 'Critical', 92, 'Operational', ['Develop BCP and DRP immediately. Conduct BIA. Define RTO/RPO.'], 'Critical'),
      ('BCP never tested.',                                                                      'NON-COMPLIANT',  'High',     'High',     78, 'Operational', ['Conduct tabletop exercise immediately. Full failover test annually.'], 'Major'),
      ('RTO: 72 hours. No automated failover. No offsite backup testing.',                       'PARTIALLY COMPLIANT', 'High',  'High',    65, 'Operational', ['Reduce RTO. Implement automated failover. Test backup restoration quarterly.'], 'Major'),
      ('BCP documented and tested annually. RTO 4 hours. Backup restoration validated.',        'COMPLIANT',      'Low',      'Low',      8,  'Operational', ['Maintain current posture. Consider ISO 22301 certification.'], 'Observation'),
      ('No ICT continuity plan for critical systems per DORA requirements.',                    'NON-COMPLIANT',  'High',     'Critical', 82, 'Regulatory',  ['Develop DORA-compliant ICT continuity plan. Test at least annually.'], 'Critical'),
    ]
  },

  'Audit Logging & Monitoring': {
    'frameworks': 'ISO 27001 A.8.15/A.8.16, NIST AU-2/AU-12, SOC 2 CC7.2, PCI DSS Req.10, HIPAA §164.312(b)',
    'controls': ['ISO 27001 A.8.15 (Logging)', 'ISO 27001 A.8.16 (Monitoring)', 'NIST AU-12', 'PCI DSS 10.2', 'SOC 2 CC7.2'],
    'scenarios': [
      ('No centralised logging. Logs stored locally on each server.',                            'NON-COMPLIANT',  'High',     'High',     79, 'Security',    ['Deploy SIEM. Centralise all logs. Retain per policy (PCI: 1yr, HIPAA: 6yr).'], 'Major'),
      ('Logs retained for 7 days. PCI DSS scoped environment.',                                 'NON-COMPLIANT',  'High',     'Critical', 84, 'Regulatory',  ['PCI DSS Req.10.7: retain 12 months (3 months online). Implement immediately.'], 'Critical'),
      ('SIEM deployed. No alerting rules configured. No SOC review of alerts.',                 'PARTIALLY COMPLIANT', 'High',  'High',    68, 'Security',   ['Configure use-case-based alerting. Assign SOC analysts. Define SLAs for alert triage.'], 'Major'),
      ('SIEM with tuned alerts, 24/7 SOC, 12-month retention, integrity protection on logs.',  'COMPLIANT',      'Low',      'Low',      6,  'Security',    ['Maintain current posture. Review alert coverage quarterly.'], 'Observation'),
      ('Admin actions not logged on production database.',                                       'NON-COMPLIANT',  'High',     'Critical', 81, 'Security',    ['Enable database activity monitoring. Log all DDL/DML by privileged users.'], 'Critical'),
    ]
  },

  'Change Management': {
    'frameworks': 'ISO 27001 A.8.32, NIST CM-3/CM-4, SOC 2 CC8.1, PCI DSS Req.6.5, COBIT BAI06',
    'controls': ['ISO 27001 A.8.32 (Change Management)', 'NIST CM-3', 'SOC 2 CC8.1', 'PCI DSS 6.5'],
    'scenarios': [
      ('Production changes deployed without approval or testing.',                               'NON-COMPLIANT',  'Critical', 'Critical', 90, 'Operational', ['Implement formal change approval process. No direct production access without CAB approval.'], 'Critical'),
      ('No change management process. Developers deploy directly to production.',               'NON-COMPLIANT',  'Critical', 'Critical', 93, 'Security',    ['Implement change management policy, CAB, and CI/CD pipeline with gates.'], 'Critical'),
      ('Change process exists. Emergency changes bypass approval 40% of the time.',             'PARTIALLY COMPLIANT', 'High',  'High',    67, 'Operational', ['Restrict emergency change path. Require post-hoc approval within 24hrs. Audit bypass rate.'], 'Major'),
      ('Formal CAB, documented RFC process, rollback plans tested, monthly change reviews.',    'COMPLIANT',      'Low',      'Low',      7,  'Operational', ['Maintain current posture. Consider automation of low-risk standard changes.'], 'Observation'),
    ]
  },

  'Security Awareness & Training': {
    'frameworks': 'ISO 27001 A.6.3, NIST AT-2/AT-3, CIS Control 14, SAMA CSF Domain 3.3, HIPAA §164.308(a)(5)',
    'controls': ['ISO 27001 A.6.3 (Awareness)', 'NIST AT-2', 'CIS Control 14', 'SAMA CSF 3.3'],
    'scenarios': [
      ('No security awareness training program exists.',                                         'NON-COMPLIANT',  'High',     'High',     76, 'Operational', ['Implement mandatory annual training. Include phishing simulation. Track completion.'], 'Major'),
      ('Training conducted once at onboarding only. No refreshers.',                            'NON-COMPLIANT',  'Medium',   'Medium',   58, 'Operational', ['Mandate annual refresher training. Add role-based training for privileged users.'], 'Minor'),
      ('Annual training mandatory. 60% completion rate. No phishing simulation.',               'PARTIALLY COMPLIANT', 'Medium', 'Medium', 49, 'Operational', ['Drive completion to 95%+. Add quarterly phishing simulations. Track click rates.'], 'Minor'),
      ('Annual training, 98% completion, quarterly phishing sims, role-based training for admins.','COMPLIANT',   'Low',      'Low',      5,  'Operational', ['Maintain current posture. Add advanced training for high-risk roles.'], 'Observation'),
    ]
  },

  'Cloud Security': {
    'frameworks': 'CSA CCM v4, ISO 27001 A.5.23, NIST 800-53 SC/AC, CIS Benchmarks, FedRAMP, SOC 2 CC6',
    'controls': ['CSA CCM DSP-07 (Data Security)', 'CSA CCM IAM-02', 'CIS Benchmark Level 2', 'NIST SC-28'],
    'scenarios': [
      ('S3 buckets publicly readable. No bucket policy review.',                                 'NON-COMPLIANT',  'Critical', 'Critical', 95, 'Security',    ['Block all public access immediately. Audit all buckets. Enable S3 Block Public Access.'], 'Critical'),
      ('No cloud security posture management (CSPM) tool deployed.',                            'NON-COMPLIANT',  'High',     'High',     74, 'Security',    ['Deploy CSPM (AWS Security Hub, Defender for Cloud). Remediate critical findings first.'], 'Major'),
      ('Root AWS account used for daily operations. No MFA on root.',                           'NON-COMPLIANT',  'Critical', 'Critical', 96, 'Security',    ['Enable MFA on root immediately. Create individual IAM users. Lock root account.'], 'Critical'),
      ('CSPM deployed. CIS Level 1 compliant. No GuardDuty/threat detection.',                  'PARTIALLY COMPLIANT', 'Medium', 'High', 55, 'Security',   ['Enable threat detection (GuardDuty/Sentinel). Route alerts to SIEM.'], 'Minor'),
      ('CSPM, CIS L2, GuardDuty, immutable logging, RBAC, encryption everywhere.',             'COMPLIANT',      'Low',      'Low',      5,  'Security',    ['Maintain current posture. Consider FedRAMP alignment if serving government.'], 'Observation'),
    ]
  },

}

# ── Response builder ───────────────────────────────────────────────────────
def build_response(domain, scenario_text, verdict, likelihood, impact, score, category, remediation, finding):
    data = DOMAINS[domain]
    remed_text = '\n'.join(f'{i+1}. {r}' for i, r in enumerate(remediation))
    conf = max(10, 100 - score) if verdict == 'COMPLIANT' else min(97, score + random.randint(-4, 4))

    return f"""## Executive Summary
{verdict} with respect to {domain} controls. Scenario: {scenario_text.strip('.')}. Risk score: {score}/100.

## Applicable Frameworks & Controls
Frameworks: {data['frameworks']}
Key controls assessed:
{chr(10).join('- ' + c for c in data['controls'])}

## Analysis
The described practice was evaluated against the applicable framework requirements. {'The control gap directly violates mandatory requirements.' if verdict == 'NON-COMPLIANT' else 'Partial conformity identified — core controls present but gaps remain.' if verdict == 'PARTIALLY COMPLIANT' else 'The control implementation satisfies applicable requirements.'}

## Evidence Found
- Practice described: {scenario_text}

## Evidence Missing
{'- Compensating controls not evidenced\n- Documentary evidence not provided\n- Audit trail not demonstrated' if verdict != 'COMPLIANT' else '- No material evidence gaps identified'}

## Risk Assessment
Likelihood: {likelihood}
Impact: {impact}
Risk Score: {score}/100
Risk Category: {category}

## Compliance Determination
{verdict}

## Cross-Framework Mapping
| Primary | Equivalent |
|---------|------------|
| {data['controls'][0]} | {data['frameworks'].split(',')[0].strip()} |
| {data['controls'][1] if len(data['controls'])>1 else data['controls'][0]} | {data['frameworks'].split(',')[1].strip() if ',' in data['frameworks'] else ''} |

## Recommended Remediation
{remed_text}

## Audit Impact
Finding: {finding}
{'Immediate corrective action required before next audit.' if finding in ['Critical','Major'] else 'Document and monitor at next scheduled review.'}

## Confidence Score
{conf}/100"""


# ── Build SFT dataset ──────────────────────────────────────────────────────
def build_prompt(scenario_text, response):
    return (
        f"<s>[INST] <<SYS>>\n{CIE_SYSTEM_PROMPT}\n<</SYS>>\n\n"
        f"Assess the following practice for compliance:\n\n{scenario_text} [/INST] "
        f"{response}</s>"
    )

sft_records = []
dpo_records = []

for domain, data in DOMAINS.items():
    for scenario in data['scenarios']:
        txt, verdict, likelihood, impact, score, category, remediation, finding = scenario
        response = build_response(domain, txt, verdict, likelihood, impact, score, category, remediation, finding)
        prompt   = build_prompt(txt, response)
        sft_records.append({'text': prompt, 'domain': domain, 'verdict': verdict})

        # ── DPO pair: chosen = correct structured response, rejected = vague chatbot reply ──
        rejected = (
            f"Based on what you've described, there might be some compliance concerns here. "
            f"You should probably look into {domain.lower()} requirements and "
            f"make sure you're following best practices. It's important to stay compliant."
        )
        dpo_records.append({
            'prompt':   f"<s>[INST] <<SYS>>\n{CIE_SYSTEM_PROMPT}\n<</SYS>>\n\nAssess: {txt} [/INST] ",
            'chosen':   response,
            'rejected': rejected,
        })

# ── Augment to 2,000+ by generating variations ─────────────────────────────
PREFIXES = [
    'Our policy states: ', 'The vendor contract requires: ', 'Current practice: ',
    'The audit found: ', 'Control evidence shows: ', 'The SOP states: ',
    'During assessment it was found that: ', 'The CISO confirmed: ',
]
SUFFIXES = [
    ' No exceptions documented.', ' This applies to all systems.',
    ' Applies to production environment.', ' Policy not enforced in practice.',
    ' Documented but not verified.', ' Verified by internal audit.',
]

base_count = len(sft_records)
target     = 2000
attempts   = 0

while len(sft_records) < target and attempts < 10000:
    attempts += 1
    domain, data = random.choice(list(DOMAINS.items()))
    scenario = random.choice(data['scenarios'])
    txt, verdict, likelihood, impact, score, category, remediation, finding = scenario

    # Vary the scenario text
    varied = random.choice(PREFIXES) + txt.lower() + random.choice(SUFFIXES)
    score_v = max(1, min(99, score + random.randint(-8, 8)))

    response = build_response(domain, varied, verdict, likelihood, impact, score_v, category, remediation, finding)
    prompt   = build_prompt(varied, response)
    sft_records.append({'text': prompt, 'domain': domain, 'verdict': verdict})

# ── Insufficient evidence examples (constitutional training) ───────────────
INSUFF_SCENARIOS = [
    'We have a security policy.',
    'Access controls are in place.',
    'We are GDPR compliant.',
    'Encryption is implemented.',
    'We conduct regular audits.',
    'The vendor is certified.',
    'Our team is trained on compliance.',
    'We have an incident response plan.',
]
for s in INSUFF_SCENARIOS:
    resp = f"""## Executive Summary
INSUFFICIENT EVIDENCE. The statement '{s}' does not provide enough detail to make a compliance determination.

## Analysis
A compliance determination requires specific, verifiable evidence. Broad statements without supporting documentation cannot be assessed.

## Evidence Found
- Assertion: {s}

## Evidence Missing
- Specific control implementation details
- Documentary evidence (policies, logs, certificates)
- Scope definition
- Date of last review or test
- Responsible owner

## Compliance Determination
INSUFFICIENT EVIDENCE

## Recommended Remediation
1. Provide specific documentation supporting the claim.
2. Define scope, owner, and last-reviewed date.
3. Submit evidence for formal assessment.

## Audit Impact
Finding: Observation
Auditors cannot accept verbal assertions without supporting evidence.

## Confidence Score
N/A — determination not possible without evidence"""
    sft_records.append({'text': build_prompt(s, resp), 'domain': 'Constitutional', 'verdict': 'INSUFFICIENT EVIDENCE'})
    dpo_records.append({
        'prompt': f"<s>[INST] <<SYS>>\n{CIE_SYSTEM_PROMPT}\n<</SYS>>\n\nAssess: {s} [/INST] ",
        'chosen': resp,
        'rejected': f"Based on what you described, you appear to be compliant. Good job!",
    })

random.shuffle(sft_records)
random.shuffle(dpo_records)

print(f'SFT records  : {len(sft_records):,}')
print(f'DPO pairs    : {len(dpo_records):,}')
print(f'Domains      : {len(DOMAINS)} + constitutional')
verdict_counts = {}
for r in sft_records:
    verdict_counts[r['verdict']] = verdict_counts.get(r['verdict'], 0) + 1
print('Verdict dist :', verdict_counts)

# Save to JSONL
with open('cie_sft_dataset.jsonl', 'w') as f:
    for r in sft_records:
        f.write(json.dumps(r) + '\n')
with open('cie_dpo_dataset.jsonl', 'w') as f:
    for r in dpo_records:
        f.write(json.dumps(r) + '\n')
print('Datasets saved to JSONL.')


## ✅ Step 5: Dataset Validation & Stats

In [ ]:
from datasets import Dataset
import numpy as np

# Load and validate
sft_ds_raw = [json.loads(l) for l in open('cie_sft_dataset.jsonl')]
dpo_ds_raw = [json.loads(l) for l in open('cie_dpo_dataset.jsonl')]

# Token length distribution
lengths = [len(r['text'].split()) for r in sft_ds_raw]
print('=== SFT Dataset ===')
print(f'Total examples : {len(sft_ds_raw):,}')
print(f'Avg length     : {np.mean(lengths):.0f} words')
print(f'Max length     : {np.max(lengths):,} words')
print(f'Min length     : {np.min(lengths):,} words')
print(f'P95 length     : {np.percentile(lengths, 95):.0f} words')

# Validate required sections
REQUIRED_SECTIONS = [
    'Executive Summary', 'Applicable Frameworks', 'Analysis',
    'Evidence Found', 'Risk Assessment', 'Compliance Determination',
    'Recommended Remediation', 'Audit Impact', 'Confidence Score'
]
missing_sections = 0
for r in sft_ds_raw:
    for s in REQUIRED_SECTIONS:
        if s not in r['text']:
            missing_sections += 1
            break

print(f'\nSection validation')
print(f'Records with all required sections : {len(sft_ds_raw) - missing_sections:,}')
print(f'Records missing sections           : {missing_sections}')

# Domain coverage
print('\nDomain coverage:')
from collections import Counter
domain_counts = Counter(r['domain'] for r in sft_ds_raw)
for domain, count in sorted(domain_counts.items(), key=lambda x: -x[1]):
    print(f'  {domain:<35} {count:>5}')

print(f'\nDPO pairs: {len(dpo_ds_raw):,}')
print('\n✅ Validation complete. Dataset is production-ready.')

# Build HF datasets
sft_hf = Dataset.from_list([{'text': r['text']} for r in sft_ds_raw])
split   = sft_hf.train_test_split(test_size=0.1, seed=42)
train_ds  = split['train']
eval_ds   = split['test']
dpo_hf    = Dataset.from_list(dpo_ds_raw)
dpo_split = dpo_hf.train_test_split(test_size=0.1, seed=42)
dpo_train = dpo_split['train']
dpo_eval  = dpo_split['test']

print(f'\nSFT train : {len(train_ds):,} | SFT eval : {len(eval_ds):,}')
print(f'DPO train : {len(dpo_train):,} | DPO eval : {len(dpo_eval):,}')


## 🤖 Step 6: Load Base Model with QLoRA (4-bit NF4)

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

torch.cuda.empty_cache()
gc.collect()

MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

# 4-bit NF4 quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16 if is_t4 else torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token, trust_remote_code=True)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=hf_token,
    trust_remote_code=True,
)
model.config.use_cache       = False
model.config.pretraining_tp  = 1

total_params     = sum(p.numel() for p in model.parameters())
print(f'Model loaded   : {total_params/1e9:.2f}B parameters')
print(f'VRAM used      : {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'VRAM free      : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB')

# LoRA config — all attention + FFN layers for maximum domain adaptation
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params : {trainable:,} ({100*trainable/total_params:.3f}% of total)')


## 🏋️ Step 7: Phase 1 — Supervised Fine-Tuning (SFT)

In [ ]:
import time
from transformers import TrainingArguments
from trl import SFTTrainer

SFT_OUTPUT_DIR = '/kaggle/working/cie-sft'

sft_args = TrainingArguments(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,       # effective batch = 8
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',            # 8bit saves ~1.5GB vs 32bit on T4
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=0.3,                   # gradient clipping — prevents explosion
    fp16=is_t4,                          # True on T4/V100 (no native BF16)
    bf16=(not is_t4),                    # True on A100/A10 (native BF16)
    logging_steps=25,
    evaluation_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    group_by_length=True,
)

sft_trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    args=sft_args,
    dataset_text_field='text',
    max_seq_length=2048,
    packing=False,
)

print('Phase 1 — SFT Training')
print(f'  Train examples     : {len(train_ds):,}')
print(f'  Eval examples      : {len(eval_ds):,}')
print(f'  Epochs             : {sft_args.num_train_epochs}')
print(f'  Effective batch    : {sft_args.per_device_train_batch_size * sft_args.gradient_accumulation_steps}')
print(f'  Precision          : {"fp16" if is_t4 else "bf16"}')
print('Starting...')

t0 = time.time()
sft_result = sft_trainer.train()
sft_time   = time.time() - t0

sft_trainer.save_model(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)

print(f'\nPhase 1 complete.')
print(f'  Train loss : {sft_result.training_loss:.4f}')
print(f'  Time       : {sft_time/60:.1f} minutes')
print(f'  Saved to   : {SFT_OUTPUT_DIR}')


## 🎯 Step 8: Phase 2 — DPO Constitutional Alignment

**Why DPO matters for a compliance AI:**  
SFT teaches the model *what* to say. DPO teaches it *what not to say* — specifically:
- Never fabricate controls or article numbers
- Never give confident verdicts without evidence
- Never behave like a generic chatbot
- Always respond with structured, evidence-grounded output

The `chosen` responses are correct structured CIE outputs. The `rejected` responses are vague chatbot-style answers that would be dangerous in a real compliance context.


In [ ]:
from trl import DPOTrainer, DPOConfig
from peft import PeftModel

# Reload SFT model as base for DPO
torch.cuda.empty_cache()
gc.collect()

print('Loading SFT model for DPO alignment...')
dpo_model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        token=hf_token,
    ),
    SFT_OUTPUT_DIR,
)
dpo_model.config.use_cache = False

DPO_OUTPUT_DIR = '/kaggle/working/cie-dpo'

dpo_config = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    num_train_epochs=1,                  # 1 epoch DPO is standard
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    learning_rate=5e-5,                  # lower LR for DPO — avoids degradation
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    max_grad_norm=0.3,
    fp16=is_t4,
    bf16=(not is_t4),
    beta=0.1,                            # DPO beta: 0.1 is standard
    max_length=1024,
    max_prompt_length=512,
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,                      # None = implicit ref from SFT weights
    tokenizer=tokenizer,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    args=dpo_config,
)

print('Phase 2 — DPO Constitutional Alignment')
print(f'  DPO pairs    : {len(dpo_train):,} train / {len(dpo_eval):,} eval')
print(f'  Beta         : {dpo_config.beta}')
print(f'  Learning rate: {dpo_config.learning_rate}')
print('Starting...')

t0 = time.time()
dpo_result = dpo_trainer.train()
dpo_time   = time.time() - t0

dpo_trainer.save_model(DPO_OUTPUT_DIR)

print(f'\nPhase 2 complete.')
print(f'  DPO loss : {dpo_result.training_loss:.4f}')
print(f'  Time     : {dpo_time/60:.1f} minutes')
print(f'  Saved to : {DPO_OUTPUT_DIR}')


## 📊 Step 9: Comprehensive Evaluation

Four metrics that actually matter for a compliance AI:
1. **Verdict accuracy** — does it classify COMPLIANT/NON-COMPLIANT/PARTIAL/INSUFFICIENT correctly?
2. **ROUGE-L** — how structurally similar is the output to reference responses?
3. **Hallucination rate** — does it invent framework names or control IDs that don't exist?
4. **Section coverage** — does every response contain all 9 mandatory CIE sections?


In [ ]:
from rouge_score import rouge_scorer
import re

# Load final model
from peft import PeftModel
from transformers import AutoModelForCausalLM

torch.cuda.empty_cache()
gc.collect()

print('Loading final DPO-aligned model for evaluation...')
eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto', token=hf_token
)
eval_model = PeftModel.from_pretrained(eval_base, DPO_OUTPUT_DIR)
eval_model.eval()

def cie_inference(text, max_new_tokens=512):
    prompt = (
        f"<s>[INST] <<SYS>>\n{CIE_SYSTEM_PROMPT}\n<</SYS>>\n\n"
        f"Assess the following practice for compliance:\n\n{text} [/INST] "
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.1, top_p=0.9, do_sample=True,
            pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.1
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# ── Eval test set (held-out, not in training data) ──────────────────────────
EVAL_CASES = [
    # (input, expected_verdict, expected_finding)
    ('No penetration test has been conducted in 3 years. PCI DSS environment.',
     'NON-COMPLIANT', 'Critical'),
    ('RBAC enforced, MFA mandatory, quarterly access reviews, same-day deprovisioning.',
     'COMPLIANT', 'Observation'),
    ('Personal data transferred to US vendor. No SCCs or adequacy decision in place.',
     'NON-COMPLIANT', 'Critical'),
    ('Annual DR test. RTO 2hrs. No offsite backup validation.',
     'PARTIALLY COMPLIANT', 'Minor'),
    ('We are compliant.',
     'INSUFFICIENT EVIDENCE', 'Observation'),
    ('AES-256 at rest, TLS 1.3 in transit, documented key rotation every 12 months.',
     'COMPLIANT', 'Observation'),
    ('Production deployments occur without CAB approval. No rollback plan.',
     'NON-COMPLIANT', 'Critical'),
    ('Security training completed at onboarding. No annual refresher. 70% staff trained.',
     'PARTIALLY COMPLIANT', 'Minor'),
]

scorer     = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
VALID_CONTROLS = [
    'ISO 27001', 'NIST', 'SOC 2', 'CIS Control', 'GDPR', 'PCI DSS',
    'HIPAA', 'DORA', 'NIS2', 'EU AI Act', 'ISO 42001', 'SAMA', 'CCPA',
]
REQUIRED_SECTIONS = [
    'Executive Summary', 'Applicable Frameworks', 'Analysis',
    'Evidence Found', 'Risk Assessment', 'Compliance Determination',
    'Recommended Remediation', 'Audit Impact', 'Confidence Score'
]

results = []
print(f'Running evaluation on {len(EVAL_CASES)} held-out cases...\n')

for i, (scenario, expected_verdict, expected_finding) in enumerate(EVAL_CASES):
    output = cie_inference(scenario)

    # Verdict accuracy
    verdicts = ['NON-COMPLIANT','PARTIALLY COMPLIANT','COMPLIANT','INSUFFICIENT EVIDENCE']
    detected = next((v for v in verdicts if v in output.upper()), 'UNKNOWN')
    verdict_correct = (detected == expected_verdict)

    # ROUGE-L (vs a reference response from training data)
    ref_response = build_response(
        list(DOMAINS.keys())[i % len(DOMAINS)],
        scenario, expected_verdict, 'High', 'High', 70, 'Security',
        ['Remediate immediately.'], expected_finding
    )
    rouge_l = scorer.score(ref_response, output)['rougeL'].fmeasure

    # Hallucination check — verify cited frameworks are real
    cited_frameworks = re.findall(r'(ISO \d+|NIST [A-Z0-9-]+|SOC 2|CIS Control|GDPR|PCI DSS|HIPAA|DORA|NIS2)', output)
    hallucinated = [f for f in cited_frameworks if not any(v in f for v in VALID_CONTROLS)]
    hallucination_rate = len(hallucinated) / max(len(cited_frameworks), 1)

    # Section coverage
    sections_present = sum(1 for s in REQUIRED_SECTIONS if s in output)
    section_score    = sections_present / len(REQUIRED_SECTIONS)

    results.append({
        'scenario': scenario[:60],
        'expected': expected_verdict,
        'detected': detected,
        'verdict_correct': verdict_correct,
        'rouge_l': rouge_l,
        'hallucination_rate': hallucination_rate,
        'section_score': section_score,
        'sections_present': sections_present,
    })

    status = '✅' if verdict_correct else '❌'
    print(f'[{i+1}] {status} Expected: {expected_verdict:<25} Got: {detected}')
    print(f'     ROUGE-L: {rouge_l:.3f}  |  Hallucinations: {len(hallucinated)}  |  Sections: {sections_present}/{len(REQUIRED_SECTIONS)}')
    print()

# ── Aggregate metrics ──────────────────────────────────────────────────────
accuracy         = sum(r['verdict_correct'] for r in results) / len(results)
avg_rouge        = sum(r['rouge_l'] for r in results) / len(results)
avg_hallucination= sum(r['hallucination_rate'] for r in results) / len(results)
avg_section      = sum(r['section_score'] for r in results) / len(results)

print('=' * 55)
print('EVALUATION RESULTS')
print('=' * 55)
print(f'Verdict Accuracy    : {accuracy*100:.1f}%')
print(f'Avg ROUGE-L         : {avg_rouge:.3f}')
print(f'Hallucination Rate  : {avg_hallucination*100:.1f}% (lower is better)')
print(f'Section Coverage    : {avg_section*100:.1f}%')
print()

# Production thresholds
THRESHOLDS = {
    'Verdict Accuracy': (accuracy, 0.80, '≥80% to pass'),
    'ROUGE-L':          (avg_rouge, 0.30, '≥0.30 to pass'),
    'Hallucination':    (1-avg_hallucination, 0.95, '≤5% hallucination to pass'),
    'Section Coverage': (avg_section, 0.90, '≥90% to pass'),
}
all_pass = True
for metric, (value, threshold, desc) in THRESHOLDS.items():
    passed = value >= threshold
    all_pass = all_pass and passed
    print(f'  {"✅" if passed else "❌"} {metric:<22} {value:.3f}  ({desc})')

print()
print('OVERALL:', '✅ PRODUCTION READY' if all_pass else '❌ NEEDS MORE TRAINING')


## 🔴 Step 10: Adversarial Testing

Testing the model against inputs specifically designed to expose failure modes:
- **Fabrication traps** — asking about made-up framework names
- **Confidence traps** — vague inputs that should trigger INSUFFICIENT EVIDENCE
- **Contradiction traps** — conflicting evidence in the same scenario
- **Scope traps** — mixing frameworks that don't apply


In [ ]:
ADVERSARIAL_CASES = [
    {
        'name': 'Fabrication trap — fake framework',
        'input': 'We are certified under ISO 99999 and XYZ-SEC-2024. Are we compliant?',
        'expect': 'should NOT cite ISO 99999 or XYZ-SEC as real frameworks',
        'fail_if': ['ISO 99999', 'XYZ-SEC-2024'],
    },
    {
        'name': 'Confidence trap — vague assertion',
        'input': 'We have good security.',
        'expect': 'INSUFFICIENT EVIDENCE',
        'fail_if': ['COMPLIANT', 'NON-COMPLIANT'],
    },
    {
        'name': 'Contradiction trap — mixed evidence',
        'input': 'MFA is enforced on all systems. However, the CEO and CFO are exempt from MFA for convenience.',
        'expect': 'PARTIALLY COMPLIANT or NON-COMPLIANT — exemptions invalidate the control',
        'fail_if': ['## Compliance Determination\nCOMPLIANT'],
    },
    {
        'name': 'Chatbot regression — generic question',
        'input': 'What is GDPR?',
        'expect': 'structured response, not a chatbot definition',
        'fail_if': ['GDPR stands for', 'General Data Protection Regulation is a'],
    },
    {
        'name': 'Scope trap — inapplicable framework',
        'input': 'Our mobile game app processes no financial data. Are we PCI DSS compliant?',
        'expect': 'should note PCI DSS is not applicable or out of scope',
        'fail_if': ['NON-COMPLIANT\n##'],
    },
]

print('Adversarial Evaluation\n' + '='*50)
adv_pass = 0

for case in ADVERSARIAL_CASES:
    output = cie_inference(case['input'], max_new_tokens=400)
    failed = [f for f in case['fail_if'] if f.upper() in output.upper()]
    passed = len(failed) == 0
    adv_pass += int(passed)

    print(f'\n[{"✅ PASS" if passed else "❌ FAIL"}] {case["name"]}')
    print(f'  Expected : {case["expect"]}')
    if failed:
        print(f'  Triggered: {failed}')
    print(f'  Output   : {output[:200]}...')

print(f'\nAdversarial score: {adv_pass}/{len(ADVERSARIAL_CASES)}')
print('✅ Model is robust' if adv_pass >= 4 else '⚠️  Model needs DPO refinement')


## 💾 Step 11: Merge LoRA Weights & Export

In [ ]:
from peft import AutoPeftModelForCausalLM

MERGED_DIR = '/kaggle/working/cie-production-final'

torch.cuda.empty_cache()
gc.collect()

print('Merging DPO adapter weights into base model...')
merged = AutoPeftModelForCausalLM.from_pretrained(
    DPO_OUTPUT_DIR,
    torch_dtype=torch.float16 if is_t4 else torch.bfloat16,
    device_map='auto',
    token=hf_token,
)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print(f'Merged model saved to: {MERGED_DIR}')
print()
print('Deployment options:')
print('  1. HF Hub    : push_to_hub(REPO_NAME)  — uncomment cell below')
print('  2. Ollama    : convert to GGUF with llama.cpp, then: ollama create cie -f Modelfile')
print('  3. vLLM      : vllm serve cie-production-final  (production API serving)')
print('  4. Next step : add RAG layer with pinecone/chroma indexing regulatory text')


In [ ]:
# ── OPTIONAL: Push to Hugging Face Hub ──────────────────────────────────
# Uncomment and set your repo name to publish

# REPO_NAME = 'your-username/cie-compliance-7b-production'
# merged.push_to_hub(REPO_NAME, token=hf_token, private=True)
# tokenizer.push_to_hub(REPO_NAME, token=hf_token, private=True)
# print(f'Published: https://huggingface.co/{REPO_NAME}')


## 📋 Step 12: Full Production Training Report

In [ ]:
total_time = sft_time + dpo_time

report = {
    'model': {
        'base'              : MODEL_ID,
        'method'            : 'QLoRA (4-bit NF4) + SFT + DPO',
        'lora_rank'         : 16,
        'lora_alpha'        : 32,
        'target_modules'    : 'q/k/v/o/gate/up/down proj (all attention + FFN)',
    },
    'dataset': {
        'sft_examples'      : len(sft_ds_raw),
        'dpo_pairs'         : len(dpo_ds_raw),
        'domains'           : list(DOMAINS.keys()) + ['Constitutional'],
        'domain_count'      : len(DOMAINS) + 1,
    },
    'training': {
        'sft_epochs'        : 3,
        'dpo_epochs'        : 1,
        'sft_loss'          : round(sft_result.training_loss, 4),
        'dpo_loss'          : round(dpo_result.training_loss, 4),
        'total_time_mins'   : round(total_time/60, 1),
        'precision'         : 'fp16' if is_t4 else 'bf16',
        'optimizer'         : 'paged_adamw_8bit',
    },
    'evaluation': {
        'verdict_accuracy'  : f'{accuracy*100:.1f}%',
        'rouge_l'           : f'{avg_rouge:.3f}',
        'hallucination_rate': f'{avg_hallucination*100:.1f}%',
        'section_coverage'  : f'{avg_section*100:.1f}%',
        'adversarial_score' : f'{adv_pass}/{len(ADVERSARIAL_CASES)}',
        'production_ready'  : all_pass,
    },
    'output': {
        'sft_model'         : SFT_OUTPUT_DIR,
        'dpo_model'         : DPO_OUTPUT_DIR,
        'merged_model'      : MERGED_DIR,
    }
}

print(json.dumps(report, indent=2))

with open('/kaggle/working/cie_training_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('\nReport saved: /kaggle/working/cie_training_report.json')
print(f'\nStatus: {"✅ PRODUCTION READY" if all_pass else "❌ REVIEW METRICS ABOVE"}')


---
## 🗺️ What's Next (Post-Kaggle)

| Phase | What | Why |
|-------|------|-----|
| **RAG** | Index full regulatory text (GDPR, ISO 27001, NIST 800-53) into a vector DB | Grounds answers in actual source material — eliminates residual hallucination |
| **Serving** | Deploy with vLLM + FastAPI | Production-grade throughput and latency |
| **UI** | Build a compliance assessment portal on top of the API | Makes the model accessible to non-technical auditors |
| **Monitoring** | Log all queries + verdicts, track drift | Catch model degradation in production |
| **v2 training** | Fine-tune on real audit findings + your own case data | The model gets smarter from every real assessment it sees |

*CIE Production Model — 2-phase training (SFT + DPO) · 12 domains · 2,000+ examples · Evaluated on accuracy, ROUGE, hallucination, adversarial robustness*
